# TeaLeafDiseaseBD — revision experiments

Self-contained. Does **not** import the `la` / `sc` / `fsc` / `fss` helper
modules used in the original runs.

Produces:

| Cell | Output | Paper target |
|---|---|---|
| 4 | `A_mask_perturbation.json` | **Table 11** (the lost one) |
| 5 | `B_encoder_70label.json` | **Table 2**, rerun fairly |
| 6 | `C_external_masks.json` + contact sheets | **§4.5 / Fig 4** validity |
| 7 | `D_head_stability.json` | §4.2 "same two heads every seed" |
| 8 | `E_distillation_bootstrap.json` | **Table 9** significance |
| 9 | `RECONCILIATION.md` + LaTeX tables | §4.12 |

Every JSON embeds the full `CFG` dict. That is the fix for what went wrong
with Table 11: a number can always be traced to the run that produced it.

**Runtime** ~2–3 h on a T4. Cells 8 is the long one and can be skipped.

In [ ]:
# ============================================================
# CELL 0 - Configuration, environment, dataset index
# ============================================================
# CFG is written into every output JSON. If you change anything
# here, bump RUN_TAG so old and new outputs never get confused.

!pip install -q timm opencv-python-headless

import os, json, time, shutil, hashlib, random
import numpy as np
import torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2
from scipy.fftpack import dct
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
except Exception:
    DRIVE = os.path.expanduser('~')

CFG = {
    'RUN_TAG'     : 'revision_v1',
    'SEED'        : 42,
    'TEST_FRAC'   : 0.30,      # matches Tables 7 / Fig 6. Set 0.20 to match Sec 4.1.
    'N_SHOT'      : 10,        # 10 per class = 70 labels
    'N_REPEATS'   : 5,
    'IMG_SIZE'    : 224,
    'CROP_PAD'    : 0.06,
    'ERODE_K'     : 11,
    'DILATE_K'    : 11,
    'SHIFT_FRAC'  : 0.12,
    'UAHS_POOL'   : 200,
    'UAHS_TOPK'   : 2,
    'ATTN_TOPK'   : int(0.1 * 14 * 14),   # = 19, floor NOT ceil (matches original code)
    'DATA_ROOT'   : '/content/drive/MyDrive/Tea Leaf Dataset',
    'LOCAL_DATA'  : '/tmp/tea_leaf_local',
    'OUT'         : '/content/drive/MyDrive/TeaLeaf_DINOv2_Study',
}
CFG['RESULTS_DIR'] = os.path.join(CFG['OUT'], 'results_' + CFG['RUN_TAG'])
CFG['FIGURES_DIR'] = os.path.join(CFG['OUT'], 'figures_' + CFG['RUN_TAG'])
os.makedirs(CFG['RESULTS_DIR'], exist_ok=True)
os.makedirs(CFG['FIGURES_DIR'], exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('ATTN_TOPK (kappa) =', CFG['ATTN_TOPK'], ' <- paper says ceil(0.10*196)=19; '
      'int() truncates, so the notation should be floor')

random.seed(CFG['SEED']); np.random.seed(CFG['SEED']); torch.manual_seed(CFG['SEED'])


def save_json(name, obj):
    obj = {'_cfg': CFG, '_saved': time.strftime('%Y-%m-%d %H:%M:%S'), **obj}
    p = os.path.join(CFG['RESULTS_DIR'], name)
    with open(p, 'w') as f:
        json.dump(obj, f, indent=2, default=float)
    print('saved ->', p)
    return p


# ---- copy dataset to local disk ----------------------------------
def count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

if count_files(CFG['LOCAL_DATA']) < count_files(CFG['DATA_ROOT']):
    from concurrent.futures import ThreadPoolExecutor
    tasks = []
    for root, _, files_ in os.walk(CFG['DATA_ROOT']):
        rel = os.path.relpath(root, CFG['DATA_ROOT'])
        dst_dir = CFG['LOCAL_DATA'] if rel == '.' else os.path.join(CFG['LOCAL_DATA'], rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            s, d = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(d):
                tasks.append((s, d))
    def _cp(t):
        try:
            shutil.copyfile(t[0], t[1]); return True
        except Exception:
            return False
    with ThreadPoolExecutor(max_workers=32) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc='copy'))
    print('copied', sum(ok), '/', len(tasks))
else:
    print('local cache complete:', count_files(CFG['LOCAL_DATA']), 'files')

classes = sorted(d for d in os.listdir(CFG['LOCAL_DATA'])
                 if os.path.isdir(os.path.join(CFG['LOCAL_DATA'], d)))
CLASS_TO_IDX = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
healthy_idx = [CLASS_TO_IDX[c] for c in healthy_classes]
print('classes:', classes)
print('healthy:', healthy_classes)

IMAGES = []
for c in classes:
    d = os.path.join(CFG['LOCAL_DATA'], c)
    for fn in sorted(os.listdir(d)):
        if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            IMAGES.append({'path': os.path.join(d, fn), 'cls': c,
                           'y': CLASS_TO_IDX[c],
                           'diseased': 0 if c in healthy_classes else 1})
print('indexed:', len(IMAGES), 'images')

In [ ]:
# ============================================================
# CELL 1 - Integrity pass: WHICH image is being dropped?
# ============================================================
# The paper says 13,084. Every pipeline output in the old notebooks
# says 13,083. This finds the missing one instead of guessing.

def probe_image(rec):
    # returns (ok_decode, ok_mask, note)
    try:
        img = Image.open(rec['path']).convert('RGB')
        a = np.array(img)
    except Exception as e:
        return False, False, 'decode:' + type(e).__name__
    if a.size == 0 or min(a.shape[:2]) < 8:
        return True, False, 'degenerate_size:%sx%s' % (a.shape[1], a.shape[0])
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV)
    v, s = hsv[:, :, 2], hsv[:, :, 1]
    try:
        _, mv = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        _, ms = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    except Exception as e:
        return True, False, 'otsu:' + type(e).__name__
    m = ((mv > 0) | (ms > 0)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return True, False, 'no_foreground'
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    if stats[k, cv2.CC_STAT_AREA] < 64:
        return True, False, 'foreground_too_small'
    return True, True, 'ok'


bad = []
for r in tqdm(IMAGES, desc='integrity'):
    okd, okm, note = probe_image(r)
    if not (okd and okm):
        bad.append({'path': r['path'], 'cls': r['cls'], 'note': note})

print('\ntotal indexed      :', len(IMAGES))
print('decode/mask failures:', len(bad))
for b in bad:
    print('   ', b['note'], '|', b['cls'], '|', os.path.basename(b['path']))

BAD_PATHS = {b['path'] for b in bad}
KEEP = [r for r in IMAGES if r['path'] not in BAD_PATHS]
print('\nusable (KEEP)      :', len(KEEP))
print('-> report this number in the paper, and state the exclusion if != 13084')

save_json('_integrity.json', {'n_indexed': len(IMAGES), 'n_keep': len(KEEP),
                              'failures': bad})

In [ ]:
# ============================================================
# CELL 2 - Segmentation, crop normalisation, shape descriptors
# ============================================================

def leaf_mask(bgr_or_rgb, rgb=True):
    a = bgr_or_rgb if rgb else cv2.cvtColor(bgr_or_rgb, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV)
    v, s = hsv[:, :, 2], hsv[:, :, 1]
    _, mv = cv2.threshold(v, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, ms = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = ((mv > 0) | (ms > 0)).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.ones(a.shape[:2], np.uint8)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    m = (lab == k).astype(np.uint8)
    # fill interior holes so dark lesions cannot perforate the mask
    ff = m.copy()
    h, w = m.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 1)
    return (m | (1 - ff)).astype(np.uint8)


def perturb_mask(m, kind, rng=None):
    if kind == 'clean':
        return m
    if kind == 'erode':
        k = np.ones((CFG['ERODE_K'], CFG['ERODE_K']), np.uint8)
        return cv2.erode(m, k, iterations=1)
    if kind == 'dilate':
        k = np.ones((CFG['DILATE_K'], CFG['DILATE_K']), np.uint8)
        return cv2.dilate(m, k, iterations=1)
    if kind == 'shift':
        h, w = m.shape
        dx = int(round(CFG['SHIFT_FRAC'] * w)); dy = int(round(CFG['SHIFT_FRAC'] * h))
        M = np.float32([[1, 0, dx], [0, 1, dy]])
        return cv2.warpAffine(m, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
    if kind == 'jitter':
        rng = rng or np.random.default_rng(0)
        out = m.copy()
        for op in rng.permutation(['erode', 'dilate', 'shift']):
            if rng.random() < 0.5:
                out = perturb_mask(out, str(op), rng)
        return out
    raise ValueError(kind)


def crop_to_leaf(rgb, m, pad=None):
    pad = CFG['CROP_PAD'] if pad is None else pad
    ys, xs = np.where(m > 0)
    if len(xs) < 16:
        return Image.fromarray(rgb).resize((CFG['IMG_SIZE'],) * 2)
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = x1 - x0 + 1, y1 - y0 + 1
    px, py = int(pad * bw), int(pad * bh)
    H, W = rgb.shape[:2]
    x0 = max(0, x0 - px); x1 = min(W - 1, x1 + px)
    y0 = max(0, y0 - py); y1 = min(H - 1, y1 + py)
    return Image.fromarray(rgb[y0:y1 + 1, x0:x1 + 1]).resize((CFG['IMG_SIZE'],) * 2)


def shape_descriptors(m):
    # the seven descriptors of Sec 3.3 - every pixel value discarded
    H, W = m.shape
    area = float(m.sum())
    if area < 16:
        return [0.0] * 7
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    c = max(cnts, key=cv2.contourArea)
    peri = float(cv2.arcLength(c, True))
    hull = cv2.convexHull(c)
    hull_a = float(cv2.contourArea(hull)) + 1e-6
    x, y, bw, bh = cv2.boundingRect(c)
    rel_area   = area / (H * W)
    aspect     = bw / (bh + 1e-6)
    solidity   = area / hull_a
    circ       = 4 * np.pi * area / (peri ** 2 + 1e-6)
    peri_size  = peri / (np.sqrt(area) + 1e-6)
    if len(c) >= 5:
        (_, _), (MA, ma), _ = cv2.fitEllipse(c)
        MA, ma = max(MA, ma), min(MA, ma)
        ecc = float(np.sqrt(max(0.0, 1 - (ma / (MA + 1e-6)) ** 2)))
    else:
        ecc = 0.0
    fill = area / (bw * bh + 1e-6)
    return [rel_area, aspect, solidity, circ, peri_size, ecc, fill]

SHAPE_NAMES = ['rel_area', 'aspect', 'solidity', 'circularity',
               'perimeter_to_size', 'eccentricity', 'fill_fraction']
print('shape descriptors:', SHAPE_NAMES)
print('NOTE: Algorithm 2 in the paper lists 6 and swaps in "raggedness". '
      'Make Sec 3.3 and Algorithm 2 agree with this list.')

In [ ]:
# ============================================================
# CELL 3 - Duplicate audit + reproducible group-aware split
# ============================================================

def _phash_bits(gray32):
    d = dct(dct(gray32.astype(np.float32), axis=0, norm='ortho'), axis=1, norm='ortho')
    low = d[:8, :8].copy()
    med = np.median(np.concatenate([low[0, 1:], low[1:, :].ravel()]))
    return (low > med).ravel()

def _dhash_bits(gray98):
    return (gray98[:, 1:] > gray98[:, :-1]).ravel()

def _bits_to_int(b):
    return int(''.join('1' if x else '0' for x in b), 2)

def canonical_hashes(path):
    im = Image.open(path).convert('L')
    ps, ds = [], []
    for tr in [None, Image.FLIP_LEFT_RIGHT, Image.FLIP_TOP_BOTTOM, Image.ROTATE_90,
               Image.ROTATE_180, Image.ROTATE_270, Image.TRANSPOSE, Image.TRANSVERSE]:
        v = im if tr is None else im.transpose(tr)
        ps.append(_bits_to_int(_phash_bits(np.array(v.resize((32, 32))))))
        ds.append(_bits_to_int(_dhash_bits(np.array(v.resize((9, 8))))))
    return min(ps), min(ds)

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ph, dh, mh = {}, {}, {}
for r in tqdm(KEEP, desc='hashing'):
    p, d = canonical_hashes(r['path'])
    ph.setdefault(p, []).append(r['path'])
    dh.setdefault(d, []).append(r['path'])
    mh.setdefault(md5(r['path']), []).append(r['path'])

parent = {r['path']: r['path'] for r in KEEP}
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for table in (ph, dh, mh):
    for _, group in table.items():
        for other in group[1:]:
            union(group[0], other)

clusters = {}
for r in KEEP:
    clusters.setdefault(find(r['path']), []).append(r['path'])
multi = {k: v for k, v in clusters.items() if len(v) > 1}
print('duplicate clusters (size>=2):', len(multi))
print('images inside a cluster     :', sum(len(v) for v in multi.values()))

cluster_ids = sorted(clusters.keys())
rng = np.random.default_rng(CFG['SEED'])
order = rng.permutation(len(cluster_ids))
target = int(round(CFG['TEST_FRAC'] * len(KEEP)))
test_paths, n = set(), 0
for i in order:
    if n >= target:
        break
    grp = clusters[cluster_ids[i]]
    test_paths.update(grp); n += len(grp)

TRAIN = [r for r in KEEP if r['path'] not in test_paths]
TEST  = [r for r in KEEP if r['path'] in test_paths]
straddle = sum(1 for v in clusters.values()
               if any(p in test_paths for p in v) and any(p not in test_paths for p in v))

print('\nSPLIT  seed=%d  test_frac=%.2f' % (CFG['SEED'], CFG['TEST_FRAC']))
print('  train %d | test %d | total %d' % (len(TRAIN), len(TEST), len(KEEP)))
print('  test diseased %.1f%%' % (100 * np.mean([r['diseased'] for r in TEST])))
print('  clusters straddling the split:', straddle)
print('\n>>> USE n = %d AS THE TEST-SET SIZE IN EVERY TABLE CAPTION <<<' % len(TEST))

save_json('_split.json', {
    'n_train': len(TRAIN), 'n_test': len(TEST), 'n_total': len(KEEP),
    'n_dup_clusters': len(multi), 'straddle': straddle,
    'test_diseased_frac': float(np.mean([r['diseased'] for r in TEST])),
    'test_paths': sorted(test_paths)})

In [ ]:
# ============================================================
# CELL 4 - Encoders + feature extraction helper
# ============================================================
import timm

NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
TO_T = T.Compose([T.ToTensor(), NORM])

_ATTN = {}

def load_dinov1():
    p = os.path.join(CFG['OUT'], 'checkpoints', 'dino_vits16.pth')
    if not os.path.exists(p):
        os.makedirs(os.path.dirname(p), exist_ok=True)
        os.system('wget -q -O %s https://dl.fbaipublicfiles.com/dino/'
                  'dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth' % p)
    m = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
    ck = torch.load(p, map_location='cpu')
    sd = {k.replace('module.', '').replace('backbone.', '').replace('encoder.', ''): v
          for k, v in ck.items() if 'head' not in k}
    m.load_state_dict(sd, strict=False)
    blk = m.blocks[-1].attn
    def fwd(x, **kw):
        B, N, C = x.shape
        qkv = blk.qkv(x).reshape(B, N, 3, blk.num_heads, C // blk.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        sc = (C // blk.num_heads) ** -0.5
        cls_attn = ((q[:, :, 0:1, :] @ k.transpose(-2, -1)) * sc).softmax(-1)
        _ATTN['cls'] = cls_attn[:, :, 0, 1:].detach().cpu()
        full = ((q @ k.transpose(-2, -1)) * sc).softmax(-1)
        x = (full @ v).transpose(1, 2).reshape(B, N, C)
        return blk.proj_drop(blk.proj(x))
    blk.forward = fwd
    return m.eval().to(device)

ENCODERS = {
    'DINOv1 ViT-S/16': load_dinov1,
    'DINOv2 ViT-S/14': lambda: timm.create_model(
        'vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).eval().to(device),
    'CLIP ViT-B/16'  : lambda: timm.create_model(
        'vit_base_patch16_clip_224.openai', pretrained=True, num_classes=0).eval().to(device),
}


class CropDS(Dataset):
    # applies leaf_mask -> perturb -> crop -> resize, on the fly
    def __init__(self, recs, kind='clean', seed=0):
        self.recs, self.kind, self.seed = recs, kind, seed
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        rgb = np.array(Image.open(r['path']).convert('RGB'))
        m = leaf_mask(rgb)
        m = perturb_mask(m, self.kind, np.random.default_rng(self.seed + i))
        return TO_T(crop_to_leaf(rgb, m)), r['y'], r['diseased']


@torch.no_grad()
def extract(model, recs, kind='clean', bs=64, desc='feat'):
    dl = DataLoader(CropDS(recs, kind), batch_size=bs, num_workers=4, shuffle=False)
    F, Y, D = [], [], []
    for x, y, d in tqdm(dl, desc=desc):
        f = model(x.to(device))
        f = torch.nn.functional.normalize(f, dim=1)
        F.append(f.cpu().numpy()); Y.append(y.numpy()); D.append(d.numpy())
    return np.concatenate(F), np.concatenate(Y), np.concatenate(D)


@torch.no_grad()
def shape_matrix(recs, kind='clean', desc='shape'):
    out = []
    for i, r in enumerate(tqdm(recs, desc=desc)):
        rgb = np.array(Image.open(r['path']).convert('RGB'))
        m = perturb_mask(leaf_mask(rgb), kind, np.random.default_rng(i))
        out.append(shape_descriptors(m))
    return np.asarray(out, np.float32)

print('encoders available:', list(ENCODERS))

In [ ]:
# ============================================================
# CELL 5 - EXPERIMENT A  ->  TABLE 11 (mask perturbation)
# ============================================================
# Probe trained ONCE on clean crops from N_SHOT labels per class.
# Held fixed. Test set re-evaluated under each mask perturbation.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

enc = ENCODERS['DINOv1 ViT-S/16']()

Xtr, ytr, dtr = extract(enc, TRAIN, 'clean', desc='train clean')
CONDITIONS = ['clean', 'erode', 'dilate', 'shift', 'jitter']
Xte = {}
for k in CONDITIONS:
    Xte[k], yte, dte = extract(enc, TEST, k, desc='test ' + k)


def fit_probe(X, y, idx):
    p = LogisticRegression(max_iter=3000, C=1.0, multi_class='multinomial')
    p.fit(X[idx], y[idx])
    return p

def binary_score(probe, X):
    P = probe.predict_proba(X)
    cols = [i for i, c in enumerate(probe.classes_) if c in healthy_idx]
    return 1.0 - P[:, cols].sum(1)


rowsA = {k: [] for k in CONDITIONS}
for rep in range(CFG['N_REPEATS']):
    rng = np.random.default_rng(CFG['SEED'] * 1000 + rep)
    idx = np.concatenate([rng.choice(np.where(ytr == c)[0], CFG['N_SHOT'], replace=False)
                          for c in range(len(classes))])
    probe = fit_probe(Xtr, ytr, idx)
    for k in CONDITIONS:
        rowsA[k].append(roc_auc_score(dte, binary_score(probe, Xte[k])))

print('\n' + '=' * 62)
print('TABLE 11  screening AUC under perturbed leaf masks')
print('  probe held fixed | n_test = %d | %d repeats' % (len(TEST), CFG['N_REPEATS']))
print('=' * 62)
LABEL = {'clean': 'Clean (no perturbation)', 'erode': 'Erode (under-segment)',
         'dilate': 'Dilate (over-segment)',
         'shift': 'Shift %d%% (misplace)' % int(100 * CFG['SHIFT_FRAC']),
         'jitter': 'Random jitter'}
tabA = {}
for k in CONDITIONS:
    m, s = float(np.mean(rowsA[k])), float(np.std(rowsA[k]))
    tabA[k] = {'mean': m, 'std': s, 'runs': rowsA[k]}
    print('  %-26s %.3f +/- %.3f' % (LABEL[k], m, s))
print('=' * 62)

save_json('A_mask_perturbation.json',
          {'n_test': len(TEST), 'kernel': CFG['ERODE_K'],
           'shift_frac': CFG['SHIFT_FRAC'], 'table': tabA})

In [ ]:
# ============================================================
# CELL 6 - EXPERIMENT B  ->  TABLE 2, rerun fairly
# ============================================================
# The published Table 2 ranked encoders by the CONFOUNDED attention
# score. This ranks them the way the deployed system actually uses
# them: 70-label probe on leaf-crop-normalised features.

resB = {}
for name, ctor in ENCODERS.items():
    m = ctor()
    Xa, ya, da = extract(m, TRAIN, 'clean', desc=name + ' train')
    Xb, yb, db = extract(m, TEST,  'clean', desc=name + ' test')
    aucs, accs = [], []
    for rep in range(CFG['N_REPEATS']):
        rng = np.random.default_rng(CFG['SEED'] * 1000 + rep)
        idx = np.concatenate([rng.choice(np.where(ya == c)[0], CFG['N_SHOT'], replace=False)
                              for c in range(len(classes))])
        p = fit_probe(Xa, ya, idx)
        aucs.append(roc_auc_score(db, binary_score(p, Xb)))
        accs.append(float((p.predict(Xb) == yb).mean()))
    resB[name] = {'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
                  'acc7_mean': float(np.mean(accs)), 'runs': aucs}
    del m; torch.cuda.empty_cache()
    print('  %-20s AUC %.4f +/- %.4f | 7-class acc %.4f'
          % (name, resB[name]['auc_mean'], resB[name]['auc_std'], resB[name]['acc7_mean']))

print('\n' + '=' * 62)
print('TABLE 2 (revised): frozen encoder, %d labels, crop-normalised' % (CFG['N_SHOT'] * 7))
print('=' * 62)
for k, v in sorted(resB.items(), key=lambda kv: -kv[1]['auc_mean']):
    print('  %-20s %.3f +/- %.3f' % (k, v['auc_mean'], v['auc_std']))
print('=' * 62)
print('If DINOv1 no longer wins, say so and either switch or report both.')

save_json('B_encoder_70label.json', {'n_labels': CFG['N_SHOT'] * 7, 'results': resB})

In [ ]:
# ============================================================
# CELL 7 - EXPERIMENT C  ->  external-dataset MASK QUALITY
# ============================================================
# The old run printed "segmentation 100%" for every external set,
# but that only means a mask was RETURNED. The recorded leaf areas
# (PlantDoc 0.86, RoCoLe 0.69 vs ours 0.27) suggest the mask is
# most of the frame there. If so, the shape descriptors on those
# datasets are near-constant and the RQ3 null result is mechanical.

import matplotlib.pyplot as plt

EXTERNAL = {   # <-- point these at your local copies; missing ones are skipped
    'PlantVillage'    : '/content/drive/MyDrive/datasets/plantvillage',
    'PlantDoc'        : '/content/drive/MyDrive/datasets/plantdoc',
    'RoCoLe'          : '/content/drive/MyDrive/datasets/rocole',
    'RiceLeafDiseaseBD': '/content/drive/MyDrive/datasets/riceleafbd',
    'TeaLeafDiseaseBD': CFG['LOCAL_DATA'],
}
N_SAMPLE, N_SHEET = 300, 12

def list_images(root, limit=4000):
    out = []
    for r, _, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                out.append(os.path.join(r, f))
                if len(out) >= limit:
                    return out
    return out

resC = {}
for name, root in EXTERNAL.items():
    if not os.path.isdir(root):
        print('skip (not found):', name); continue
    paths = list_images(root)
    if not paths:
        print('skip (empty):', name); continue
    rng = np.random.default_rng(CFG['SEED'])
    sel = [paths[i] for i in rng.choice(len(paths), min(N_SAMPLE, len(paths)), replace=False)]
    areas, sols, fills = [], [], []
    for p in tqdm(sel, desc=name):
        rgb = np.array(Image.open(p).convert('RGB'))
        m = leaf_mask(rgb)
        d = shape_descriptors(m)
        areas.append(d[0]); sols.append(d[2]); fills.append(d[6])
    areas = np.array(areas)
    resC[name] = {'n': len(sel),
                  'area_mean': float(areas.mean()), 'area_median': float(np.median(areas)),
                  'area_std': float(areas.std()),
                  'frac_area_gt_0.5': float((areas > 0.5).mean()),
                  'frac_area_gt_0.8': float((areas > 0.8).mean()),
                  'solidity_mean': float(np.mean(sols)),
                  'fill_mean': float(np.mean(fills))}

    fig, ax = plt.subplots(3, 4, figsize=(13, 10))
    for a, p in zip(ax.ravel(), sel[:N_SHEET]):
        rgb = np.array(Image.open(p).convert('RGB'))
        m = leaf_mask(rgb)
        ov = rgb.copy()
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, cnts, -1, (255, 0, 0), max(2, rgb.shape[0] // 200))
        a.imshow(ov); a.axis('off')
        a.set_title('area=%.2f' % (m.sum() / m.size), fontsize=8)
    fig.suptitle('%s - Otsu leaf masks (red contour)' % name, fontsize=13)
    fig.tight_layout()
    fp = os.path.join(CFG['FIGURES_DIR'], 'masks_%s.png' % name.replace('/', '_'))
    fig.savefig(fp, dpi=130, bbox_inches='tight'); plt.close(fig)
    print('   contact sheet ->', fp)

print('\n' + '=' * 78)
print('%-20s %6s %8s %8s %10s %10s' % ('dataset', 'n', 'area', 'median', '>0.5', '>0.8'))
print('-' * 78)
for k, v in resC.items():
    print('%-20s %6d %8.3f %8.3f %10.2f %10.2f'
          % (k, v['n'], v['area_mean'], v['area_median'],
             v['frac_area_gt_0.5'], v['frac_area_gt_0.8']))
print('=' * 78)
print('READ: if an external set has mean area >> 0.5 and most images >0.5,')
print('the mask is the frame, not a leaf. Then its shape-only AUC and its')
print('dAUC are uninformative, and Sec 4.5 / Sec 5.4 need the scope caveat:')
print('the diagnostic assumes reliable single-object segmentation.')

save_json('C_external_masks.json', {'datasets': resC})

In [ ]:
# ============================================================
# CELL 8 - EXPERIMENT D  ->  UAHS head-selection stability
# ============================================================
# Sec 4.2 claims "UAHS selects the same two heads in every seed".
# The old notebooks show [0,1] in one place and [1,3] in another.

@torch.no_grad()
def uahs_heads(recs, pool=None, seed=0):
    pool = pool or CFG['UAHS_POOL']
    rng = np.random.default_rng(seed)
    sel = [recs[i] for i in rng.choice(len(recs), min(pool, len(recs)), replace=False)]
    dl = DataLoader(CropDS(sel, 'clean'), batch_size=32, num_workers=4)
    ents = []
    for x, _, _ in dl:
        _ = enc(x.to(device))
        a = _ATTN['cls'].numpy()                    # (B, heads, 196)
        a = a / (a.sum(-1, keepdims=True) + 1e-12)
        ents.append(-(a * np.log(a + 1e-12)).sum(-1))
    E = np.concatenate(ents, 0)                     # (N, heads)
    u = E.std(0)
    return list(np.argsort(-u)[:CFG['UAHS_TOPK']]), u.tolist()

sel_by_seed = {}
for s in range(5):
    h, u = uahs_heads(TRAIN, seed=CFG['SEED'] * 1000 + s)
    sel_by_seed[s] = {'heads': [int(x) for x in h], 'uahs': u}
    print('seed %d -> heads %s   uahs %s' % (s, h, np.round(u, 4).tolist()))

sets = {tuple(sorted(v['heads'])) for v in sel_by_seed.values()}
stable = len(sets) == 1
print('\nSTABLE ACROSS SEEDS:', stable, '|', sets)
print('-> if False, soften Sec 4.2 to report the modal selection and its frequency.')

save_json('D_head_stability.json', {'per_seed': sel_by_seed, 'stable': stable,
                                    'distinct_selections': [list(x) for x in sets]})

In [ ]:
# ============================================================
# CELL 9 - EXPERIMENT E  ->  Table 9 significance   (LONG, ~40 min)
# ============================================================
# Settles whether probe+distillation really beats labels-only, by
# bootstrapping the AUC difference on paired per-image test scores.
RUN_E = True

if RUN_E:
    import torchvision.models as tvm
    import torch.optim as optim

    class ArrDS(Dataset):
        def __init__(self, recs, labels, train):
            self.r, self.l, self.train = recs, labels, train
            self.aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                                  T.RandomRotation(20), T.ColorJitter(.2, .2, .2)])
        def __len__(self): return len(self.r)
        def __getitem__(self, i):
            rec = self.r[i]
            rgb = np.array(Image.open(rec['path']).convert('RGB'))
            im = crop_to_leaf(rgb, leaf_mask(rgb))
            if self.train: im = self.aug(im)
            return TO_T(im), int(self.l[i])

    def train_student(recs, labels, epochs=30, bs=64):
        m = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
        m = m.to(device)
        dl = DataLoader(ArrDS(recs, labels, True), batch_size=bs, shuffle=True, num_workers=4)
        opt = optim.Adam(m.parameters(), lr=1e-3)
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        lf = nn.CrossEntropyLoss()
        for _ in tqdm(range(epochs), desc='student'):
            m.train()
            for x, y in dl:
                opt.zero_grad()
                lf(m(x.to(device)), y.to(device)).backward(); opt.step()
            sch.step()
        return m

    @torch.no_grad()
    def student_scores(m, recs):
        m.eval()
        dl = DataLoader(ArrDS(recs, np.zeros(len(recs)), False), batch_size=128, num_workers=4)
        out = [torch.softmax(m(x.to(device)), 1)[:, 1].cpu().numpy() for x, _ in dl]
        return np.concatenate(out)

    S = {'labels_only': [], 'probe_distil': []}
    for rep in range(3):
        rng = np.random.default_rng(CFG['SEED'] * 1000 + rep)
        idx = np.concatenate([rng.choice(np.where(ytr == c)[0], CFG['N_SHOT'], replace=False)
                              for c in range(len(classes))])
        seed_recs = [TRAIN[i] for i in idx]
        seed_bin  = dtr[idx]
        S['labels_only'].append(student_scores(train_student(seed_recs, seed_bin), TEST))
        probe = fit_probe(Xtr, ytr, idx)
        pseudo = (binary_score(probe, Xtr) > 0.5).astype(int)
        pseudo[idx] = seed_bin
        S['probe_distil'].append(student_scores(train_student(TRAIN, pseudo), TEST))

    def boot_diff(a, b, y, n=2000, seed=0):
        rng = np.random.default_rng(seed); d = []
        for _ in range(n):
            i = rng.integers(0, len(y), len(y))
            if len(np.unique(y[i])) < 2: continue
            d.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
        return np.percentile(d, [2.5, 50, 97.5])

    a = np.mean(S['probe_distil'], 0); b = np.mean(S['labels_only'], 0)
    lo, mid, hi = boot_diff(a, b, dte)
    print('\n' + '=' * 62)
    print('TABLE 9 significance (paired bootstrap, 2000 resamples)')
    print('  probe+distil AUC : %.4f' % roc_auc_score(dte, a))
    print('  labels-only  AUC : %.4f' % roc_auc_score(dte, b))
    print('  difference       : %+.4f  95%% CI [%+.4f, %+.4f]' % (mid, lo, hi))
    print('  CI excludes zero :', bool(lo > 0))
    print('=' * 62)
    print('If the CI includes zero, drop "small but consistent gain" and say so.')

    save_json('E_distillation_bootstrap.json',
              {'auc_probe_distil': float(roc_auc_score(dte, a)),
               'auc_labels_only': float(roc_auc_score(dte, b)),
               'diff_median': float(mid), 'ci95': [float(lo), float(hi)],
               'ci_excludes_zero': bool(lo > 0)})

In [ ]:
# ============================================================
# CELL 10 - Reconciliation report + LaTeX-ready tables
# ============================================================
L = []
A = lambda s='': L.append(s)

A('# Reconciliation report  (run tag: %s)' % CFG['RUN_TAG'])
A()
A('| quantity | value | paper says | action |')
A('|---|---|---|---|')
A('| kappa | %d | 19 via ceil() | change ceil -> floor in Sec 3.4 |' % CFG['ATTN_TOPK'])
A('| images indexed | %d | 13,084 | - |' % len(IMAGES))
A('| images usable | %d | 13,084 | state exclusion if different |' % len(KEEP))
A('| test-set n | %d | 2,616 / 2,621 / 3,924 | use this everywhere |' % len(TEST))
A('| test diseased | %.1f%% | 86%% | - |' % (100 * np.mean([r['diseased'] for r in TEST])))
A('| test_frac | %.2f | not stated | state it in Sec 3.2 |' % CFG['TEST_FRAC'])
A()
A('## Table 11 - mask perturbation')
A()
A('```latex')
A('\\begin{tabular}{lc}')
A('\\toprule')
A('Mask perturbation & AUC \\\\')
A('\\midrule')
for k in CONDITIONS:
    A('%s & %.3f $\\pm$ %.3f \\\\' % (LABEL[k], tabA[k]['mean'], tabA[k]['std']))
A('\\bottomrule')
A('\\end{tabular}')
A('```')
A()
A('Caption must state: probe held fixed, n = %d, %d repeats, '
  'erosion/dilation %dx%d kernel, shift %d%% of frame.'
  % (len(TEST), CFG['N_REPEATS'], CFG['ERODE_K'], CFG['ERODE_K'],
     int(100 * CFG['SHIFT_FRAC'])))
A()
A('## Table 2 - encoder choice, rerun at %d labels' % (CFG['N_SHOT'] * 7))
A()
A('```latex')
A('\\begin{tabular}{lc}')
A('\\toprule')
A('Frozen encoder & Binary AUC \\\\')
A('\\midrule')
for k, v in sorted(resB.items(), key=lambda kv: -kv[1]['auc_mean']):
    A('%s & %.3f $\\pm$ %.3f \\\\' % (k, v['auc_mean'], v['auc_std']))
A('\\bottomrule')
A('\\end{tabular}')
A('```')
A()
A('## External mask quality (Sec 4.5 validity)')
A()
A('| dataset | n | mean area | frac > 0.5 | frac > 0.8 |')
A('|---|---|---|---|---|')
for k, v in resC.items():
    A('| %s | %d | %.3f | %.2f | %.2f |'
      % (k, v['n'], v['area_mean'], v['frac_area_gt_0.5'], v['frac_area_gt_0.8']))
A()
A('## UAHS head stability')
A()
A('Selections: %s -> stable = %s' % ([v['heads'] for v in sel_by_seed.values()], stable))
A()
A('## Still to fix by hand (no experiment needed)')
A()
for s in [
  'Table 8 caption: "Missed" is 1 - sensitivity, NOT the disease rate among cleared leaves.',
  'Line ~555 and ~664: same correction in prose.',
  'Fig 6 caption: matrices are from the single median-AUC seed, not "computed over five seeds".',
  'Sec 4.8: Table 6 is the PROBE under clutter, not the deployed student.',
  'Sec 4.9 line ~528: deployed-student std is over THREE seeds, not five.',
  'Sec 3.5 / Algorithm 2: reconcile C1-C5 vs C1-C3 numbering.',
  'Sec 3.3 / Algorithm 2: same seven shape descriptors in both.',
  'Rename one of the two "corrected pipeline"s.',
  'Sec 4.12: add the shape-control AUCs (0.834 n=1400, 0.792 full set) to the reconciliation.',
  'Ref [2] Lien & Lai vs "Islam et al." mismatch; Otsu -> IEEE TSMC 9(1):62-66; fix [16].',
  'Disclose in Sec 4.5 that RiceLeafDiseaseBD [26] is your own dataset.',
]:
    A('- ' + s)

txt = '\n'.join(L)
p = os.path.join(CFG['RESULTS_DIR'], 'RECONCILIATION.md')
open(p, 'w').write(txt)
print(txt)
print('\nwritten ->', p)